# 04 — Text analysis of ACLED `notes` (course-aligned)

Applies the POLI3148 Lecture-7 / Sessions-2-3 toolkit — TF-IDF word-frequency comparison, VADER sentiment over time, and LDA topic modelling — to the 26,977 ACLED `notes` source-narratives in the AES core slice.

Idioms taken directly from `session_2_wordfreq_topics.ipynb` and `session_3_sentiment_classification.ipynb`. Random seed 42 throughout for reproducibility.

Outputs:
- `docs/figs/fig_06_wordfreq.html` — top distinguishing words pre vs post Wagner (Mali, civilian-targeted)
- `docs/figs/fig_07_sentiment.html` — monthly VADER compound sentiment trajectories per AES country
- `docs/figs/fig_08_topics.html` — LDA K=8 topic prevalence by phase
- `docs/figs/pyldavis_mali.html` — interactive pyLDAvis (graceful fallback if pyLDAvis unavailable)
- adds 8 keys to `data/final_stats.json`

In [1]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
np.random.seed(42)

# Portable paths: notebook lives in code/, data lives in ../data, figures in ../docs/figs.
DATA = Path('../data').resolve()
FIGS = Path('../docs/figs').resolve()
FIGS.mkdir(parents=True, exist_ok=True)
STATS_PATH = DATA / 'final_stats.json'

aes = pd.read_parquet(DATA / 'acled_clean.parquet')
print('AES core notes available:', aes.notes.notna().sum(), '/', len(aes))

AES core notes available: 26977 / 26977


## 1. Light text-cleaning helper

spaCy is heavy and brings no analytical value over a careful regex/stoplist on ACLED's terse, structured notes (the dataset is itself event-coded, not free prose). We follow the same domain-stopword pattern as `session_2_wordfreq_topics.ipynb`, but use sklearn's English stopword list and a regex tokenizer for speed; the pipeline is otherwise identical.

In [2]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Domain stopwords — every ACLED notes string includes these and they swamp signal.
domain_stopwords = {
    'mali', 'malian', 'burkina', 'faso', 'burkinabe', 'niger', 'nigerien',
    'civilian', 'civilians', 'killed', 'kill', 'killing', 'killings', 'death', 'deaths',
    'people', 'person', 'persons', 'individual', 'individuals',
    'attack', 'attacked', 'attacking', 'event', 'events',
    'reported', 'reportedly', 'according', 'source', 'sources',
    'fatality', 'fatalities', 'fatal', 'death', 'die', 'died',
    'al', 'el', 'le', 'la', 'de', 'du', 'des', 'and', 'or', 'on', 'at', 'in', 'to', 'of',
    'said', 'say', 'says', 'told', 'told', 'states', 'state',
    'one', 'two', 'three', 'four', 'five', 'six', 'several', 'many',
    'including', 'include', 'included', 'others', 'other', 'another',
    'date', 'day', 'days', 'week', 'weeks', 'month', 'months', 'year', 'years',
    'around', 'approximately', 'least', 'about', 'after', 'before',
    'unknown', 'identified', 'unidentified', 'name', 'named', 'names',
    'forces', 'force', 'group', 'groups', 'armed', 'arms', 'arm',
    'between', 'amid',
    'note', 'notes', 'noted', 'reports', 'report', 'reporting',
    'man', 'men', 'woman', 'women', 'child', 'children',
}

stoplist = ENGLISH_STOP_WORDS.union(domain_stopwords)
TOK_RE = re.compile(r'[A-Za-z]{3,}')

def tokenize(text):
    if not isinstance(text, str):
        return []
    return [w.lower() for w in TOK_RE.findall(text)
            if w.lower() not in stoplist and len(w) >= 3]

# Tokenize all AES notes once (fast: simple regex, ~26k docs)
aes = aes.copy()
aes['tokens'] = aes['notes'].apply(tokenize)
print('avg tokens / note:', aes.tokens.apply(len).mean().round(1))
print('sample tokens (first row):', aes.tokens.iloc[0][:25])

avg tokens / note: 16.5
sample tokens (first row): ['looting', 'july', 'gunmen', 'stripped', 'local', 'director', 'national', 'broadcaster', 'ortm', 'driver', 'belongings', 'seized', 'vehicle', 'timbuktu']


## 2. Figure 6 — Top distinguishing words, Mali civilian-targeted, pre vs post Wagner

Course-aligned idiom: build TWO TF-IDF matrices (pre and post), compute log-odds-ratio of relative term frequencies between the two phases, plot the top 20 distinguishing words per phase as a horizontal-bar small-multiple.

In [3]:
mali_civ = aes[(aes.country == 'Mali') & aes.civilian_targeted].copy()
mali_civ['phase'] = np.where(mali_civ.post_russian_arrival, 'post-Wagner', 'pre-Wagner')
print(mali_civ.phase.value_counts())

# Build vocab from union
def identity(x):
    return x

vec = TfidfVectorizer(
    tokenizer=identity,
    preprocessor=lambda x: x,
    lowercase=False,
    min_df=10,
    max_df=0.5,
)
tfidf_mat = vec.fit_transform(mali_civ['tokens'])
vocab = vec.get_feature_names_out()

phase_arr = mali_civ['phase'].values
is_pre  = phase_arr == 'pre-Wagner'
is_post = phase_arr == 'post-Wagner'

# Log-odds-with-prior-style score: (rate_post + a) / (rate_pre + a)
alpha = 1e-3
rate_pre  = np.asarray(tfidf_mat[is_pre].mean(axis=0)).ravel()
rate_post = np.asarray(tfidf_mat[is_post].mean(axis=0)).ravel()
log_ratio = np.log((rate_post + alpha) / (rate_pre + alpha))

scores = pd.DataFrame({'word': vocab, 'rate_pre': rate_pre, 'rate_post': rate_post, 'log_ratio': log_ratio})
top_post = scores.sort_values('log_ratio', ascending=False).head(20)
top_pre  = scores.sort_values('log_ratio', ascending=True).head(20).iloc[::-1]

print('Top post-Wagner-distinctive words:')
print(top_post[['word', 'log_ratio']].to_string(index=False))
print('\nTop pre-Wagner-distinctive words:')
print(top_pre[['word', 'log_ratio']].to_string(index=False))

phase
post-Wagner    2557
pre-Wagner     1575
Name: count, dtype: int64
Top post-Wagner-distinctive words:
       word  log_ratio
     wagner   4.013543
       fama   3.864921
      sahel   3.028636
mercenaries   2.837316
     patrol   2.709866
        air   2.225908
  operation   1.942042
    injured   1.920169
      drone   1.894247
      based   1.885953
     strike   1.788449
  tidermene   1.772149
        hit   1.722097
 airstrikes   1.695936
  conducted   1.674480
    carried   1.637417
  supported   1.595582
   location   1.558512
  shepherds   1.543703
       nara   1.523778

Top pre-Wagner-distinctive words:
        word  log_ratio
       bamba  -1.527087
      device  -1.565317
   dioungani  -1.648806
    executed  -1.659590
    madougou  -1.729269
        dead  -1.732186
       dozos  -1.825989
      gunmen  -1.827835
  presumably  -1.869649
        army  -1.927838
   motorbike  -1.939855
      opened  -1.951572
    wounding  -2.027186
  motorbikes  -2.050555
   villagers  -

### Render Figure 6 — distinctive-word horizontal bars

Two-panel plotly bar chart: left panel (red) = top 20 post-Wagner-distinctive words by log-odds, right panel (blue) = top 20 pre-Wagner-distinctive words. Hover any bar for the exact log-ratio.

In [4]:
fig06 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Post-Wagner distinctive (positive log-odds)',
                    'Pre-Wagner distinctive (negative log-odds)'),
    horizontal_spacing=0.18,
)
fig06.add_trace(
    go.Bar(
        x=top_post['log_ratio'],
        y=top_post['word'],
        orientation='h',
        marker=dict(color='#8B0000'),
        hovertemplate='<b>%{y}</b><br>log-ratio post/pre = %{x:.3f}<extra></extra>',
        name='post-Wagner',
    ),
    row=1, col=1,
)
fig06.add_trace(
    go.Bar(
        x=top_pre['log_ratio'],
        y=top_pre['word'],
        orientation='h',
        marker=dict(color='#1f77b4'),
        hovertemplate='<b>%{y}</b><br>log-ratio post/pre = %{x:.3f}<extra></extra>',
        name='pre-Wagner',
    ),
    row=1, col=2,
)
fig06.update_layout(
    title=dict(text='Figure 6 — Words that most distinguish post-Wagner from pre-Wagner Mali civilian-targeting notes (TF-IDF log-ratio)',
               font=dict(size=14)),
    showlegend=False,
    height=620, margin=dict(t=80, l=20, r=20, b=40),
    paper_bgcolor='#FFFFFF', plot_bgcolor='#F4F6F9',
    font=dict(family='Segoe UI, system-ui, sans-serif', size=11),
)
fig06.update_xaxes(title_text='log( TF-IDF post / TF-IDF pre )', zeroline=True, zerolinecolor='#888')
fig06.write_html(FIGS / 'fig_06_wordfreq.html', include_plotlyjs='cdn', full_html=True)
fig06.show()
print('Saved fig_06_wordfreq.html')

Saved fig_06_wordfreq.html


## 3. Figure 7 — VADER sentiment over time, monthly mean per country

Course-aligned idiom from `session_3_sentiment_classification.ipynb`: apply `SentimentIntensityAnalyzer().polarity_scores(text)['compound']` to each notes string. Plot monthly means as a 3-panel small-multiple with the coup and Wagner-arrival vlines.

Caveat: VADER is a general-purpose lexicon (social-media-trained). On neutral-tone event-reporting prose like ACLED notes, compound scores cluster near 0 with a negative-leaning shift when violence escalates — useful as a relative metric across phases, not as an absolute sentiment thermometer.

In [5]:
analyzer = SentimentIntensityAnalyzer()

def vader_compound(text):
    if not isinstance(text, str):
        return np.nan
    return analyzer.polarity_scores(text)['compound']

# Score every AES notes row
aes['vader_compound'] = aes['notes'].apply(vader_compound)
print('VADER compound summary:')
print(aes['vader_compound'].describe().round(3))

VADER compound summary:
count    26977.000
mean        -0.506
std          0.387
min         -0.998
25%         -0.818
50%         -0.649
75%         -0.250
max          0.963
Name: vader_compound, dtype: float64


### Aggregate to monthly means and render Figure 7

Three-panel small-multiple, one row per AES country, with coup (red dashed) and Wagner-arrival (blue dashed) vertical reference lines. Values clipped to [-1, 0.3] for legibility.

In [6]:
# Monthly mean compound per country
month_sent = (aes.groupby(['country', 'year_month'])['vader_compound']
              .mean().reset_index())

# Treatment dates from final_stats
with open(STATS_PATH) as f:
    fs = json.load(f)
coup_dates = {
    'Mali': pd.Timestamp(fs['mali_first_coup']),
    'Burkina Faso': pd.Timestamp(fs['bf_first_coup']),
    'Niger': pd.Timestamp(fs['niger_first_coup']),
}
wagner_dates = {
    'Mali': pd.Timestamp(fs['mali_wagner_arrival']),
    'Burkina Faso': pd.Timestamp(fs['bf_wagner_arrival']),
    'Niger': pd.Timestamp(fs['niger_wagner_arrival']),
}

fig07 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                       subplot_titles=('Mali', 'Burkina Faso', 'Niger'))
for r, ctry in enumerate(['Mali', 'Burkina Faso', 'Niger'], start=1):
    sub = month_sent[month_sent.country == ctry].sort_values('year_month')
    fig07.add_trace(
        go.Scatter(
            x=sub['year_month'], y=sub['vader_compound'],
            mode='lines+markers', line=dict(width=1.4, color='#1B2A4A'),
            marker=dict(size=4, color='#1B2A4A'),
            name=ctry, showlegend=False,
            hovertemplate='%{x|%b %Y}<br>mean compound = %{y:.3f}<extra></extra>',
        ), row=r, col=1)
    # Zero reference
    fig07.add_hline(y=0, line=dict(color='#999', dash='dot', width=0.8), row=r, col=1)
    # Coup vline (use literal x value; avoid Plotly annotation_position bug on datetime axes in subplots)
    fig07.add_shape(type='line', x0=coup_dates[ctry], x1=coup_dates[ctry], y0=-1, y1=0.3,
                     line=dict(color='#8B0000', dash='dash', width=1.4), row=r, col=1)
    fig07.add_annotation(x=coup_dates[ctry], y=0.25, text='coup', showarrow=False,
                          font=dict(size=9, color='#8B0000'), row=r, col=1, xanchor='left')
    fig07.add_shape(type='line', x0=wagner_dates[ctry], x1=wagner_dates[ctry], y0=-1, y1=0.3,
                     line=dict(color='#2E86AB', dash='dash', width=1.4), row=r, col=1)
    fig07.add_annotation(x=wagner_dates[ctry], y=0.15, text='Wagner', showarrow=False,
                          font=dict(size=9, color='#2E86AB'), row=r, col=1, xanchor='right')

    # Bright-orange mean lines + inline midpoint labels with opaque white
    # background so they don't visually overlap the dark navy data line.
    w_ts = wagner_dates[ctry]
    pre_v = sub[sub['year_month'] < w_ts]['vader_compound']
    post_v = sub[sub['year_month'] >= w_ts]['vader_compound']
    if len(pre_v) and len(post_v):
        pre_m = pre_v.mean(); post_m = post_v.mean()
        earliest = sub['year_month'].min(); latest = sub['year_month'].max()
        MEAN_COLOR = '#FF8C00'
        fig07.add_shape(type='line', x0=earliest, x1=w_ts, y0=pre_m, y1=pre_m,
                         line=dict(color=MEAN_COLOR, width=2.5, dash='dash'),
                         row=r, col=1)
        fig07.add_shape(type='line', x0=w_ts, x1=latest, y0=post_m, y1=post_m,
                         line=dict(color=MEAN_COLOR, width=2.5, dash='dash'),
                         row=r, col=1)
        # Single consolidated textbox per panel (bottom-center) with both means
        # and orange styling matching the orange mean lines above.
        xref_p_d = 'x domain' if r == 1 else f'x{r} domain'
        yref_p_d = 'y domain' if r == 1 else f'y{r} domain'
        means_text = (
            f'<b><span style="color:{MEAN_COLOR}">Pre → post Wagner mean</span></b> '
            f'(VADER): <b>{pre_m:.2f} → {post_m:.2f}</b>'
        )
        fig07.add_annotation(xref=xref_p_d, yref=yref_p_d, x=0.5, y=0.05,
                              xanchor='center', yanchor='bottom',
                              text=means_text, showarrow=False,
                              font=dict(size=11, color='#222'),
                              bgcolor='#FFFFFF', bordercolor=MEAN_COLOR,
                              borderwidth=1.5, borderpad=5)

fig07.update_layout(
    title=dict(text='Figure 7 — VADER compound sentiment of ACLED notes, monthly mean per AES country',
               font=dict(size=14)),
    height=720, margin=dict(t=80, l=30, r=20, b=40),
    paper_bgcolor='#FFFFFF', plot_bgcolor='#F4F6F9',
    font=dict(family='Segoe UI, system-ui, sans-serif', size=11),
)
fig07.update_yaxes(title_text='mean VADER compound', range=[-1, 0.3])
fig07.write_html(FIGS / 'fig_07_sentiment.html', include_plotlyjs='cdn', full_html=True)
fig07.show()
print('Saved fig_07_sentiment.html')

Saved fig_07_sentiment.html


## 4. Figure 8 — LDA topic modelling, K=8, Mali civilian-targeted notes

Course-aligned idiom from `session_2_wordfreq_topics.ipynb`: fit `LatentDirichletAllocation(n_components=8, random_state=42, max_iter=20, learning_method='online')` on the count-vectorised tokens of Mali civilian-targeted notes. Compute topic prevalence by phase (pre vs post Wagner) by averaging document-topic distributions per phase. Plot side-by-side stacked bars.

In [7]:
cv = CountVectorizer(
    tokenizer=identity, preprocessor=lambda x: x, lowercase=False,
    min_df=10, max_df=0.5,
)
dtm = cv.fit_transform(mali_civ['tokens'])
lda_vocab = cv.get_feature_names_out()
print('DTM shape:', dtm.shape)

K = 8
lda = LatentDirichletAllocation(
    n_components=K,
    random_state=42,
    max_iter=20,
    learning_method='online',
    n_jobs=-1,
)
lda.fit(dtm)
doc_topics = lda.transform(dtm)
print('LDA fit complete; doc-topic shape:', doc_topics.shape)

# Top 6 words per topic
topic_top_words = []
for k in range(K):
    top_idx = lda.components_[k].argsort()[-6:][::-1]
    words = [lda_vocab[i] for i in top_idx]
    topic_top_words.append(words)
    print(f'Topic {k}: {", ".join(words)}')

DTM shape: (4132, 566)


LDA fit complete; doc-topic shape: (4132, 8)
Topic 0: gao, ansongo, sahel, militants, transport, aboard
Topic 1: tombouctou, gunmen, bamako, vehicle, wounded, assaulted
Topic 2: jnim, kidal, militants, abducted, likely, ied
Topic 3: militants, jnim, village, mopti, presumed, abducted
Topic 4: mopti, fulani, militiamen, village, dogon, douentza
Topic 5: gao, members, air, city, shot, carried
Topic 6: menaka, abducted, community, march, tuareg, village
Topic 7: fama, wagner, village, segou, arrested, niono


### Compute topic prevalence per phase and render Figure 8

For each topic, average the document-topic probabilities across documents in each phase (pre vs post Wagner). The grouped-bar comparison shows which topics gained or lost ground after Wagner's deployment.

In [8]:
# Topic prevalence per phase: average doc-topic distribution
phase_arr2 = mali_civ['phase'].values
prev_pre  = doc_topics[phase_arr2 == 'pre-Wagner'].mean(axis=0)
prev_post = doc_topics[phase_arr2 == 'post-Wagner'].mean(axis=0)

topic_labels = [f'T{k}: {", ".join(topic_top_words[k][:3])}' for k in range(K)]
fig08 = go.Figure()
fig08.add_trace(go.Bar(
    name='pre-Wagner', x=topic_labels, y=prev_pre,
    marker=dict(color='#1f77b4'),
    hovertemplate='<b>%{x}</b><br>pre-Wagner prevalence = %{y:.3f}<extra></extra>',
))
fig08.add_trace(go.Bar(
    name='post-Wagner', x=topic_labels, y=prev_post,
    marker=dict(color='#8B0000'),
    hovertemplate='<b>%{x}</b><br>post-Wagner prevalence = %{y:.3f}<extra></extra>',
))
fig08.update_layout(
    barmode='group',
    title=dict(text=f'Figure 8 — LDA K={K} topic prevalence in Mali civilian-targeted notes, pre vs post Wagner',
               font=dict(size=14)),
    height=560, margin=dict(t=80, l=20, r=20, b=120),
    paper_bgcolor='#FFFFFF', plot_bgcolor='#F4F6F9',
    font=dict(family='Segoe UI, system-ui, sans-serif', size=11),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(tickangle=-30),
    yaxis=dict(title='mean topic probability per document'),
)
fig08.write_html(FIGS / 'fig_08_topics.html', include_plotlyjs='cdn', full_html=True)
fig08.show()
print('Saved fig_08_topics.html')

Saved fig_08_topics.html


### Optional pyLDAvis interactive panel

Wraps the LDA model in `pyLDAvis` to give an interactive topic-distance visualisation (course-taught Session-2 idiom). Wrapped in try/except so the notebook continues even if pyLDAvis is not installed.

In [9]:
# pyLDAvis interactive — graceful fallback if unavailable
try:
    import pyLDAvis
    import pyLDAvis.lda_model
    panel = pyLDAvis.lda_model.prepare(lda, dtm, cv, mds='tsne')
    pyLDAvis.save_html(panel, str(FIGS / 'pyldavis_mali.html'))
    print('Saved pyldavis_mali.html')
except Exception as e:
    print('pyLDAvis unavailable / failed — skipping interactive visualisation. reason:', e)

Saved pyldavis_mali.html


## 5. Optional — Random Forest classifier predicting `civilian_targeted` from notes

A supervised text classifier asks: can a model trained ONLY on note text recover the civilian-targeted flag? If accuracy is far above the majority-class baseline, it confirms the textual signal is robust enough to be machine-detectable — not a cherry-picked artefact of TF-IDF or LDA.

**Methodology.** Train/test split is stratified on the target with `random_state=42`. TF-IDF is fit on TRAIN ONLY (no test-set leakage into the IDF or vocabulary). We report accuracy alongside a **majority-class baseline** (the accuracy you'd get by always predicting the dominant class), the confusion matrix, and per-class precision/recall/F1 — not just raw accuracy, which can mislead under class imbalance.


In [10]:
# Use all AES notes; target = civilian_targeted (boolean -> 0/1).
# IMPORTANT: split FIRST, then fit TF-IDF on train ONLY (avoid test-set leakage into IDF/vocab).
from sklearn.metrics import confusion_matrix, classification_report

rf_df = aes[['notes', 'civilian_targeted']].dropna(subset=['notes']).copy()
rf_df['target'] = rf_df['civilian_targeted'].astype(int)
rf_df['tokens'] = rf_df['notes'].apply(tokenize)

tokens_train, tokens_test, y_train, y_test = train_test_split(
    rf_df['tokens'].values, rf_df['target'].values,
    test_size=0.3, random_state=42, stratify=rf_df['target'].values)

rf_vec = TfidfVectorizer(tokenizer=identity, preprocessor=lambda x: x,
                         lowercase=False, min_df=20, max_df=0.5)
X_train = rf_vec.fit_transform(tokens_train)   # FIT only on train
X_test  = rf_vec.transform(tokens_test)         # TRANSFORM test
feat_names = rf_vec.get_feature_names_out()
print('feature matrix (train):', X_train.shape, '| class balance train:', np.bincount(y_train))

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
acc = metrics.accuracy_score(y_test, y_pred)
print(f'Random Forest accuracy: {acc:.4f}')
baseline_acc = max(np.bincount(y_test)) / len(y_test)
print(f'Majority-class baseline accuracy: {baseline_acc:.4f}')
print(f'RF lift over baseline: {(acc - baseline_acc) * 100:.1f} percentage points')
print('Confusion matrix:')
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred, digits=3))

imps = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
print('Top 15 features by importance:')
print(imps.head(15).round(4).to_string())

feature matrix (train): (18883, 1382) | class balance train: [12373  6510]


Random Forest accuracy: 0.9356
Majority-class baseline accuracy: 0.6553
RF lift over baseline: 28.0 percentage points
Confusion matrix:
[[5052  252]
 [ 269 2521]]
              precision    recall  f1-score   support

           0      0.949     0.952     0.951      5304
           1      0.909     0.904     0.906      2790

    accuracy                          0.936      8094
   macro avg      0.929     0.928     0.929      8094
weighted avg      0.936     0.936     0.936      8094

Top 15 features by importance:
abducted       0.0907
looting        0.0427
village        0.0318
casualties     0.0181
clashed        0.0177
fulani         0.0171
destruction    0.0128
community      0.0127
male           0.0122
assaulted      0.0109
property       0.0107
shot           0.0102
presumed       0.0099
claimed        0.0098
released       0.0097


### Logistic Regression comparator (course-aligned LR + RF pair)

Fits `LogisticRegression(class_weight='balanced')` on the SAME train/test split as the Random Forest, so the two classifiers can be compared head-to-head. The signed coefficients (positive → pushes toward `civilian_targeted = True`) feed two new keys in `final_stats.json`.

In [11]:
# Fix 3 - Logistic Regression comparator on the SAME train/test split
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', n_jobs=-1)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
lr_acc = metrics.accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression accuracy: {lr_acc:.4f}')
print('Confusion matrix (LR):')
print(metrics.confusion_matrix(y_test, y_pred_lr))
print(f'LR lift over baseline: {(lr_acc - baseline_acc) * 100:.1f} percentage points')
print(metrics.classification_report(y_test, y_pred_lr, digits=3))

# Signed coefficients: positive => pushes toward civilian_targeted=True
lr_coefs = pd.Series(lr.coef_[0], index=feat_names).sort_values(ascending=False)
lr_top_pos = lr_coefs.head(10)
lr_top_neg = lr_coefs.tail(10).iloc[::-1]
print('\nTop 10 words -> CIVILIAN-TARGETED (positive LR coef):')
print(lr_top_pos.round(3).to_string())
print('\nTop 10 words -> NON-CIVILIAN (negative LR coef):')
print(lr_top_neg.round(3).to_string())

Logistic Regression accuracy: 0.9286
Confusion matrix (LR):
[[4913  391]
 [ 187 2603]]
LR lift over baseline: 27.3 percentage points
              precision    recall  f1-score   support

           0      0.963     0.926     0.944      5304
           1      0.869     0.933     0.900      2790

    accuracy                          0.929      8094
   macro avg      0.916     0.930     0.922      8094
weighted avg      0.931     0.929     0.929      8094


Top 10 words -> CIVILIAN-TARGETED (positive LR coef):
abducted     10.127
village       6.041
assaulted     5.965
shot          4.949
fulani        4.343
cart          3.940
male          3.897
duty          3.535
community     3.512
executed      3.350

Top 10 words -> NON-CIVILIAN (negative LR coef):
looting        -16.261
clashed        -10.016
destruction     -9.229
displacement    -6.743
demonstrated    -6.222
position        -6.037
repelled        -5.497
movement        -4.943
ambushed        -4.829
property        -4.552


## 6. Append text-analysis keys to `final_stats.json`

In [12]:
from scipy import stats as scistats

with open(STATS_PATH) as f:
    fs = json.load(f)

# 1. Number of notes analysed
fs['text_n_notes_analysed'] = int(aes['notes'].notna().sum())

# 2. Mali pre/post mean compound + Welch t-test (Fix 2)
mali_aes = aes[aes.country == 'Mali']
mali_pre_vals  = mali_aes.loc[~mali_aes.post_russian_arrival, 'vader_compound'].dropna().values
mali_post_vals = mali_aes.loc[ mali_aes.post_russian_arrival, 'vader_compound'].dropna().values
mali_pre  = float(mali_pre_vals.mean())
mali_post = float(mali_post_vals.mean())
fs['text_avg_compound_pre_mali']  = round(mali_pre,  3)
fs['text_avg_compound_post_mali'] = round(mali_post, 3)
fs['text_avg_compound_diff_mali'] = round(mali_post - mali_pre, 3)

t_stat, p_val = scistats.ttest_ind(mali_post_vals, mali_pre_vals, equal_var=False)
fs['text_vader_t_stat']    = round(float(t_stat), 3)
fs['text_vader_pvalue']    = round(float(p_val),  4)
fs['text_vader_n_pre']     = int(len(mali_pre_vals))
fs['text_vader_n_post']    = int(len(mali_post_vals))
print(f'Welch t = {t_stat:.3f}, p = {p_val:.4f}, n_pre={len(mali_pre_vals)}, n_post={len(mali_post_vals)}')

# 3. Top 5 words distinctive of post-Wagner Mali civilian-targeting (post - pre)
top5 = scores.sort_values('log_ratio', ascending=False).head(5)['word'].tolist()
fs['text_top5_words_post_minus_pre_mali'] = [str(w) for w in top5]

# 4. Topic count + top-words list (one string per topic)
fs['text_topics_count'] = int(K)
fs['text_topics_top_words_post'] = [
    f'topic {k}: ' + ', '.join(topic_top_words[k]) for k in range(K)
]

# 5. RF classifier accuracy
fs['text_classifier_accuracy'] = round(float(acc), 3)
fs['text_classifier_accuracy_pct'] = int(round(float(acc) * 100))

# 6. Logistic Regression accuracy + signed top words (Fix 3)
fs['text_lr_classifier_accuracy'] = round(float(lr_acc), 3)
fs['text_lr_classifier_accuracy_pct'] = int(round(float(lr_acc) * 100))
fs['text_lr_top_pos_words'] = [str(w) for w in lr_top_pos.index.tolist()]
fs['text_lr_top_neg_words'] = [str(w) for w in lr_top_neg.index.tolist()]

with open(STATS_PATH, 'w') as f:
    json.dump(fs, f, indent=2, default=str)

print('Updated final_stats.json with text-analysis keys:')
for k in ['text_n_notes_analysed', 'text_avg_compound_pre_mali',
          'text_avg_compound_post_mali', 'text_avg_compound_diff_mali',
          'text_vader_t_stat', 'text_vader_pvalue',
          'text_vader_n_pre', 'text_vader_n_post',
          'text_top5_words_post_minus_pre_mali',
          'text_topics_count', 'text_classifier_accuracy',
          'text_lr_classifier_accuracy',
          'text_lr_top_pos_words', 'text_lr_top_neg_words']:
    print(f'  {k}: {fs[k]}')

Welch t = 2.234, p = 0.0255, n_pre=4090, n_post=6939
Updated final_stats.json with text-analysis keys:
  text_n_notes_analysed: 26977
  text_avg_compound_pre_mali: -0.54
  text_avg_compound_post_mali: -0.524
  text_avg_compound_diff_mali: 0.016
  text_vader_t_stat: 2.234
  text_vader_pvalue: 0.0255
  text_vader_n_pre: 4090
  text_vader_n_post: 6939
  text_top5_words_post_minus_pre_mali: ['wagner', 'fama', 'sahel', 'mercenaries', 'patrol']
  text_topics_count: 8
  text_classifier_accuracy: 0.936
  text_lr_classifier_accuracy: 0.929
  text_lr_top_pos_words: ['abducted', 'village', 'assaulted', 'shot', 'fulani', 'cart', 'male', 'duty', 'community', 'executed']
  text_lr_top_neg_words: ['looting', 'clashed', 'destruction', 'displacement', 'demonstrated', 'position', 'repelled', 'movement', 'ambushed', 'property']


## 7. Summary

- Word-frequency comparison (TF-IDF log-odds) saved to `fig_06_wordfreq.html`.
- VADER monthly sentiment per country saved to `fig_07_sentiment.html`.
- LDA K=8 topic prevalence by phase saved to `fig_08_topics.html`; pyLDAvis saved to `pyldavis_mali.html` if available.
- Random Forest classifier reported with accuracy and top features.
- 8 new keys appended to `final_stats.json` for prose-token resolution.